In [31]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [33]:
df = pd.read_csv('/content/IMDB Dataset.csv.zip')

print(df.head())
print(df.shape)


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [4]:
df.info()

df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,count
sentiment,
positive,25000
negative,25000


In [34]:
encoder = LabelEncoder()

df['sentiment'] = encoder.fit_transform(df['sentiment'])

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [7]:
df['sentiment'] = df['sentiment'].map({
    'negative':0,
    'positive':1
})

In [35]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [36]:
vocab_size = 10000
max_length = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [37]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_length),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [44]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6489 - loss: 0.6389 - val_accuracy: 0.6133 - val_loss: 0.6587
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6449 - loss: 0.6346 - val_accuracy: 0.6177 - val_loss: 0.6636
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.6875 - loss: 0.5928 - val_accuracy: 0.7480 - val_loss: 0.5277
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.8349 - loss: 0.4036 - val_accuracy: 0.8448 - val_loss: 0.3684
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8952 - loss: 0.2920 - val_accuracy: 0.8668 - val_loss: 0.3364
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9182 - loss: 0.2420 - val_accuracy: 0.8705 - val_loss: 0.3387
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9345 - loss: 0.2048 - val_accuracy: 0.8694 - val_loss: 0.3502
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9476 - loss: 0.1737 - val_accu

In [45]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8592 - loss: 0.6703
Test Accuracy: 0.8592000007629395


In [46]:
model.save("movie_sentiment_model.h5")

In [47]:
def predict_sentiment(review):

    seq = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        seq,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    print("\nReview:")
    print(review)

    print("\nConfidence:", round(float(prediction), 4))

    if prediction >= 0.5:
        print("Sentiment: Positive ")
    else:
        print("Sentiment: Negative ")

In [48]:
predict_sentiment(
    "This movie was absolutely fantastic. Acting and story were amazing."
)

predict_sentiment(
    "Worst movie ever. Total waste of time and money."
)


Review:
This movie was absolutely fantastic. Acting and story were amazing.

Confidence: 0.9911
Sentiment: Positive 

Review:
Worst movie ever. Total waste of time and money.

Confidence: 0.0065
Sentiment: Negative 


In [49]:
predict_sentiment("best movie ever")


Review:
best movie ever

Confidence: 0.9891
Sentiment: Positive 


In [50]:
predict_sentiment("worst movie of all time")


Review:
worst movie of all time

Confidence: 0.0169
Sentiment: Negative 


In [51]:
predict_sentiment("horrible movie")


Review:
horrible movie

Confidence: 0.0157
Sentiment: Negative 
